# Solver comparison (Step 8)

Reads the per-cube results that `scripts/evaluate.py` writes to `runs/adi_d12/`
and joins them by `(depth, cube)`. It picks up whatever evaluation files exist, so
it can be rerun as each solver finishes.

How to read it:

- **Every solver saw the same cubes.** Joining is valid because all runs used
  seed 0 and 50 cubes per depth; the loading cell checks the cube counts match.
- **IDA\* gives the optimal solution length** wherever it finished within its node
  limit, so other solvers can be compared with the optimum on the very same cube.
- **Where IDA\* did not finish, the optimum is unknown.** Gap statistics therefore
  cover only cubes IDA\* solved, which at deep scramble depths are the easier ones.
- Tables come first; no value here depends on reading a chart.

In [ ]:
import csv
import re
from collections import defaultdict
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN = ROOT / 'runs' / 'adi_d12'
OPTIMAL = 'ida_n200000'
CUBES_PATTERN = re.compile(r'eval_(?P<solver>.+)_d\d+-\d+_cubes\.csv')
SUMMARY_PATTERN = re.compile(r'eval_(?P<solver>.+)_d\d+-\d+\.csv')

# solver -> {(depth, cube): solution length, or -1 if unsolved}
lengths = defaultdict(dict)
# solver -> {depth: mean milliseconds per cube}
ms_per_cube = defaultdict(dict)
for path in sorted(RUN.glob('eval_*.csv')):
    if match := CUBES_PATTERN.fullmatch(path.name):
        with open(path) as handle:
            for row in csv.DictReader(handle):
                lengths[match['solver']][(int(row['depth']), int(row['cube']))] = int(row['length'])
    elif match := SUMMARY_PATTERN.fullmatch(path.name):
        with open(path) as handle:
            for row in csv.DictReader(handle):
                ms_per_cube[match['solver']][int(row['depth'])] = float(row['ms_per_cube'])

solvers = sorted(lengths)
depths = sorted({depth for depth, _ in lengths[OPTIMAL]})


def cubes_at(solver, depth):
    return {key: value for key, value in lengths[solver].items() if key[0] == depth}


# Changing --cubes changes which cubes are generated, so a joined comparison is
# only valid if every solver has the same number of cubes at each depth it covers.
for solver in solvers:
    for depth in {d for d, _ in lengths[solver]}:
        assert len(cubes_at(solver, depth)) == len(cubes_at(OPTIMAL, depth)), (
            f'{solver} has a different cube count from IDA* at depth {depth}'
        )

print('solvers found:', ', '.join(solvers))
print('depths:', depths)

## Solve rate by scramble depth

In [ ]:
def solve_rate(solver, depth):
    values = list(cubes_at(solver, depth).values())
    return np.mean([value >= 0 for value in values]) if values else np.nan


def percent(value):
    return '-' if np.isnan(value) else f'{value:.0%}'


column = max(len(solver) for solver in solvers) + 2
print('depth'.rjust(5) + ''.join(solver.rjust(column) for solver in solvers))
for depth in depths:
    print(str(depth).rjust(5) + ''.join(percent(solve_rate(s, depth)).rjust(column) for s in solvers))

## How far from optimal?

Extra moves compared with IDA\*'s optimal solution, on cubes both solved. A
negative gap is impossible if IDA\* is correct, so the cell asserts there are none:
that is a consistency check on the whole pipeline, not just a statistic.

In [ ]:
def optimality_gaps(solver):
    gaps = []
    for key, optimal_length in lengths[OPTIMAL].items():
        own = lengths[solver].get(key)
        if own is not None and own >= 0 and optimal_length >= 0:
            gaps.append(own - optimal_length)
    return np.array(gaps, dtype=int)


print('solver'.rjust(column), 'cubes'.rjust(6), 'optimal'.rjust(8), 'mean extra'.rjust(11), 'max extra'.rjust(10))
for solver in solvers:
    if solver == OPTIMAL:
        continue
    gaps = optimality_gaps(solver)
    if gaps.size == 0:
        continue
    assert (gaps >= 0).all(), f'{solver} found a solution shorter than the optimum'
    print(solver.rjust(column), str(gaps.size).rjust(6), f'{np.mean(gaps == 0):.0%}'.rjust(8),
          f'{gaps.mean():.2f}'.rjust(11), str(gaps.max()).rjust(10))

## Cubes solved where IDA\* ran out of nodes

For each depth, how many of the cubes IDA\* could not finish within 200,000 nodes
each other solver did finish. This is the region where a learned policy could add
something a pattern database alone cannot, within the same time scale.

In [ ]:
others = [solver for solver in solvers if solver != OPTIMAL]
print('depth'.rjust(5), 'IDA* unsolved'.rjust(14) + ''.join(solver.rjust(column) for solver in others))
for depth in depths:
    unsolved = [key for key, value in cubes_at(OPTIMAL, depth).items() if value < 0]
    if not unsolved:
        continue
    counts = []
    for solver in others:
        if not cubes_at(solver, depth):
            counts.append('-')
        else:
            counts.append(str(sum(lengths[solver].get(key, -1) >= 0 for key in unsolved)))
    print(str(depth).rjust(5), str(len(unsolved)).rjust(14) + ''.join(c.rjust(column) for c in counts))

## Paired comparisons

Outcomes are paired on the same cubes, so what matters is the discordant cubes, where
only one of two solvers finished. The exact McNemar test on those is the right test
for paired yes/no outcomes and does not rely on a large sample. Solution lengths are
compared only on cubes both solved.

- **Beam versus hybrid at equal width** is the project's controlled comparison: a test
  asserts the hybrid equals beam search when the heuristic knows nothing, so any
  difference comes from the pattern database.
- **Beam width 1000 versus hybrid width 100** asks the equal-time question: the two
  cost about the same per cube on deep cubes.

In [ ]:
from math import comb


def mcnemar_p(only_first, only_second):
    """Exact two-sided McNemar test on the discordant pairs."""
    discordant = only_first + only_second
    if discordant == 0:
        return 1.0
    tail = sum(comb(discordant, k) for k in range(min(only_first, only_second) + 1))
    return min(1.0, 2 * tail / 2 ** discordant)


def paired(first, second):
    if first not in lengths or second not in lengths:
        print(f'{first} vs {second}: waiting for results\n')
        return
    print(f'{first} vs {second}')
    print('depth'.rjust(5), 'both'.rjust(5), 'first only'.rjust(11), 'second only'.rjust(12),
          'neither'.rjust(8), 'first ms'.rjust(9), 'second ms'.rjust(10))
    first_only_total = second_only_total = 0
    differences = []
    for depth in depths:
        keys = cubes_at(first, depth).keys() & cubes_at(second, depth).keys()
        if not keys:
            continue
        both = first_only = second_only = neither = 0
        for key in keys:
            first_solved, second_solved = lengths[first][key] >= 0, lengths[second][key] >= 0
            if first_solved and second_solved:
                both += 1
                differences.append(lengths[second][key] - lengths[first][key])
            elif first_solved:
                first_only += 1
            elif second_solved:
                second_only += 1
            else:
                neither += 1
        first_only_total += first_only
        second_only_total += second_only
        print(str(depth).rjust(5), str(both).rjust(5), str(first_only).rjust(11), str(second_only).rjust(12),
              str(neither).rjust(8), f'{ms_per_cube[first].get(depth, np.nan):.1f}'.rjust(9),
              f'{ms_per_cube[second].get(depth, np.nan):.1f}'.rjust(10))
    differences = np.array(differences)
    first_time = sum(ms_per_cube[first].values()) / 1000
    second_time = sum(ms_per_cube[second].values()) / 1000
    print(f'all depths: first only {first_only_total}, second only {second_only_total}, '
          f'exact McNemar p = {mcnemar_p(first_only_total, second_only_total):.2e}')
    print(f'time, sum of per-cube means: first {first_time:.2f} s, second {second_time:.2f} s')
    print(f'on the {differences.size} cubes both solved: second shorter on {np.sum(differences < 0)}, '
          f'longer on {np.sum(differences > 0)}, mean difference {differences.mean():+.2f} moves\n')


paired('beam_w100_checkpoint', 'hybrid_w100_checkpoint')
paired('beam_w1000_checkpoint', 'hybrid_w1000_checkpoint')
paired('beam_w1000_checkpoint', 'hybrid_w100_checkpoint')

## Figures

The depth figures plot the solve-rate table above for depths 1-20, with IDA\* as a
gray reference line. Depth 50 is left off their axis, where it would compress depths
1-20, and stays in the table.

- `figures/search_width.png` -- greedy, beam width 100 and beam width 1000 in one
  ordered blue ramp, since width is an ordered quantity.
- `figures/pattern_database_effect.png` -- beam search against the hybrid at each
  width, one panel per width, so each comparison changes only the pattern database.
- `figures/cost_versus_reach.png` -- cubes solved at depths 11-20 against mean time
  per cube, one line per solver family joining its widths. Depths 11-20 were chosen
  after seeing the data, as the range where the search solvers are not all at 100%;
  the printed table repeats the values.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from chart_style import CHART, FIGURES, add_legend, plot_series, style_axes

# Validated as an ordinal ramp on this surface (light end 2.06:1).
WIDTH_RAMP = {
    'greedy_checkpoint': '#86b6ef',
    'beam_w100_checkpoint': '#3987e5',
    'beam_w1000_checkpoint': '#184f95',
}
LABELS = {
    'greedy_checkpoint': 'Greedy (width 1)',
    'beam_w100_checkpoint': 'Beam search, width 100',
    'beam_w1000_checkpoint': 'Beam search, width 1000',
    'hybrid_w100_checkpoint': 'Hybrid, width 100',
    'hybrid_w1000_checkpoint': 'Hybrid, width 1000',
    OPTIMAL: 'IDA*, 200k-node limit',
}
PLOT_DEPTHS = [depth for depth in depths if depth <= 20]


def rates(solver):
    return [solve_rate(solver, depth) for depth in PLOT_DEPTHS]


def plot_reference(ax):
    ax.plot(PLOT_DEPTHS, rates(OPTIMAL), color=CHART['ink_muted'], linewidth=1.5,
            solid_capstyle='round', label=LABELS[OPTIMAL])


def finish_depth_axes(ax, with_ylabel=True):
    ax.set_xlim(0.5, 20.5)
    ax.set_ylim(0, 1.05)
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xlabel('Scramble depth')
    if with_ylabel:
        ax.set_ylabel('Cubes solved')


fig, ax = plt.subplots(figsize=(7.5, 4.4), facecolor=CHART['surface'])
style_axes(ax)
plot_reference(ax)
for solver, color in WIDTH_RAMP.items():
    if solver in lengths:
        plot_series(ax, PLOT_DEPTHS, rates(solver), color, LABELS[solver], markevery=[-1])
finish_depth_axes(ax)
ax.set_title('Wider search solves deeper cubes', loc='left', fontsize=11)
add_legend(ax, loc='lower left')
fig.tight_layout()
fig.savefig(FIGURES / 'search_width.png', dpi=200, facecolor=CHART['surface'], bbox_inches='tight')

widths = [width for width in (100, 1000) if f'beam_w{width}_checkpoint' in lengths]
fig, axes = plt.subplots(1, len(widths), figsize=(11, 4.4), facecolor=CHART['surface'],
                         sharey=True, squeeze=False)
for index, (ax, width) in enumerate(zip(axes[0], widths)):
    style_axes(ax)
    plot_reference(ax)
    pair = ((f'beam_w{width}_checkpoint', CHART['series'][0]),
            (f'hybrid_w{width}_checkpoint', CHART['series'][1]))
    for solver, color in pair:
        if solver in lengths:
            plot_series(ax, PLOT_DEPTHS, rates(solver), color, LABELS[solver], markevery=[-1])
    finish_depth_axes(ax, with_ylabel=index == 0)
    ax.set_title(f'Width {width}', loc='left', fontsize=11)
    add_legend(ax, loc='lower left')
fig.suptitle('Beam search with and without the pattern database', x=0.01, ha='left',
             fontsize=12, color=CHART['ink'])
fig.tight_layout()
fig.savefig(FIGURES / 'pattern_database_effect.png', dpi=200, facecolor=CHART['surface'], bbox_inches='tight')

REACH_DEPTHS = range(11, 21)
BELOW_RIGHT = ((8, -12), 'left')
ABOVE_LEFT = ((-8, 8), 'right')  # beam width 1000 sits just above-left of hybrid width 100
FAMILIES = [
    ('Beam search (width 1 is greedy)', CHART['series'][0],
     [('1', 'greedy_checkpoint', BELOW_RIGHT), ('100', 'beam_w100_checkpoint', BELOW_RIGHT),
      ('1000', 'beam_w1000_checkpoint', ABOVE_LEFT)]),
    ('Hybrid with pattern database', CHART['series'][1],
     [('100', 'hybrid_w100_checkpoint', BELOW_RIGHT), ('1000', 'hybrid_w1000_checkpoint', BELOW_RIGHT)]),
]


def reach(solver):
    return np.mean([solve_rate(solver, depth) for depth in REACH_DEPTHS])


def cost(solver):
    return np.mean([ms_per_cube[solver][depth] for depth in REACH_DEPTHS])


print('solver'.rjust(column), 'solved, depths 11-20'.rjust(21), 'mean ms/cube'.rjust(13))
for solver in [OPTIMAL] + [key for _, _, members in FAMILIES for _, key, _ in members]:
    if solver in lengths and solver in ms_per_cube:
        print(solver.rjust(column), f'{reach(solver):.1%}'.rjust(21), f'{cost(solver):,.1f}'.rjust(13))

fig, ax = plt.subplots(figsize=(7.5, 4.6), facecolor=CHART['surface'])
style_axes(ax)
ax.grid(True, axis='x', which='major', color=CHART['grid'], linewidth=0.8, linestyle='-', alpha=1)
for label, color, members in FAMILIES:
    present = [member for member in members if member[1] in lengths and member[1] in ms_per_cube]
    if not present:
        continue
    xs = [cost(key) for _, key, _ in present]
    ys = [reach(key) for _, key, _ in present]
    plot_series(ax, xs, ys, color, label)
    for (width, _, (offset, align)), x, y in zip(present, xs, ys):
        ax.annotate(f'width {width}', xy=(x, y), xytext=offset, textcoords='offset points',
                    ha=align, fontsize=8.5, color=CHART['ink_secondary'])
ax.plot([cost(OPTIMAL)], [reach(OPTIMAL)], marker='o', markersize=7, color=CHART['ink_muted'],
        markeredgecolor=CHART['surface'], markeredgewidth=1.5, linestyle='none', label=LABELS[OPTIMAL])
ax.set_xscale('log')
ax.tick_params(which='minor', length=0)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_xlabel('Mean time per cube at depths 11-20 (ms, log scale)')
ax.set_ylabel('Cubes solved at depths 11-20')
ax.set_title('Cubes solved against time per cube', loc='left', fontsize=11)
add_legend(ax, loc='upper left')
fig.tight_layout()
fig.savefig(FIGURES / 'cost_versus_reach.png', dpi=200, facecolor=CHART['surface'], bbox_inches='tight')

## Does longer training help?

Greedy solve rate on the same cubes for each 10k-iteration snapshot; 60k is the
final `checkpoint.pt`. The summary uses depths 8-15, the range where no snapshot
sits at 100% or near 0%; the mean over all depths 1-20 is printed alongside to show
the conclusion does not depend on that choice. Saved to
`figures/training_length.png`.

In [ ]:
SNAPSHOTS = [
    ('20k', 'greedy_model_020000'),
    ('30k', 'greedy_model_030000'),
    ('40k', 'greedy_model_040000'),
    ('50k', 'greedy_model_050000'),
    ('60k', 'greedy_checkpoint'),
]
snapshots = [(name, key) for name, key in SNAPSHOTS if key in lengths]
BAND = range(8, 16)
ALL_DEPTHS = range(1, 21)

print('depth'.rjust(5) + ''.join(name.rjust(7) for name, _ in snapshots))
for depth in ALL_DEPTHS:
    print(str(depth).rjust(5) + ''.join(percent(solve_rate(key, depth)).rjust(7) for _, key in snapshots))

band_mean = [np.mean([solve_rate(key, depth) for depth in BAND]) for _, key in snapshots]
all_mean = [np.mean([solve_rate(key, depth) for depth in ALL_DEPTHS]) for _, key in snapshots]
print()
print('mean 8-15'.rjust(9) + ''.join(f'{value:.1%}'.rjust(8) for value in band_mean))
print('mean 1-20'.rjust(9) + ''.join(f'{value:.1%}'.rjust(8) for value in all_mean))

iterations = [int(name[:-1]) * 1000 for name, _ in snapshots]
fig, ax = plt.subplots(figsize=(7.5, 4.2), facecolor=CHART['surface'])
style_axes(ax)
plot_series(ax, iterations, band_mean, CHART['series'][0])
for index in (0, len(iterations) - 1):
    ax.annotate(f'{band_mean[index]:.0%}', xy=(iterations[index], band_mean[index]), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=9, color=CHART['ink_secondary'])
ax.set_xticks(iterations)
ax.set_xticklabels([name for name, _ in snapshots])
ax.set_xlim(15000, 65000)
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_xlabel('Training iterations')
ax.set_ylabel('Mean greedy solve rate, depths 8-15')
ax.set_title('Solve rate stops improving after about 40k iterations', loc='left', fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / 'training_length.png', dpi=200, facecolor=CHART['surface'], bbox_inches='tight')